In [ ]:
import os
import time
import json
import pandas as pd
from openai import OpenAI

# ==========================================================
# 0) Configuration (User Action Required)
# ==========================================================

# Replace with your actual API Key (OpenAI or third-party provider)
API_KEY = "YOUR_API_KEY_HERE"

# Replace with the appropriate endpoint (e.g., "https://api.openai.com/v1")
# Note: Usually includes /v1; do not end with a trailing "/"
BASE_URL = "https://your-api-endpoint.com/v1"

# Input/Output File Paths
LOOKUP_CSV = "unique_drugs_with_smiles.csv"  # Input: Must contain 'drug_name' and 'smiles' columns
OUT_CSV = "drug_desc_embedding.csv"          # Output: Results will be appended (supports resume)

# Model Selection
TEXT_MODEL = "gpt-4o-mini"
EMB_MODEL  = "text-embedding-3-small"

# Request Settings
REQUEST_SLEEP_SEC = 1.0
MAX_RETRIES = 5

# Optional: Add extra headers if required by your service provider
EXTRA_HEADERS = {
    # "x-api-key": "",
    # "Authorization": f"Bearer {API_KEY}"
}

# ==========================================================
# 1) Initialize Client
# ==========================================================
client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    default_headers=EXTRA_HEADERS if EXTRA_HEADERS else None
)

def call_with_retries(fn, *args, **kwargs):
    """
    Simple exponential backoff retry logic.
    """
    for attempt in range(MAX_RETRIES):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            wait = 2 ** attempt
            print(f"    Error: {e} | Retrying in {wait}s...")
            time.sleep(wait)
    raise RuntimeError(f"Failed after {MAX_RETRIES} retries: {fn.__name__}")

def generate_description(drug_name: str) -> str:
    """
    Generate an academic description of the drug using the LLM.
    Includes targets and pathways; uses 'Unknown' if information is missing.
    """
    prompt = (
        f"Please summarize the major function of the drug: {drug_name}. "
        "Write ONE academic paragraph. Include key targets and pathway information if known. "
        "If uncertain, explicitly say 'Unknown' rather than guessing."
    )
    resp = client.chat.completions.create(
        model=TEXT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    text = resp.choices[0].message.content or ""
    return text.strip()

def embed_text(text: str) -> list[float]:
    """
    Convert text to a vector using the embedding model.
    Returns a list of floats for easy JSON serialization.
    """
    text = (text or "").replace("\n", " ").strip()
    if not text:
        raise ValueError("Empty description text; cannot generate embedding.")
    emb = client.embeddings.create(
        model=EMB_MODEL,
        input=text
    )
    return emb.data[0].embedding

def load_done_keys(out_csv: str) -> set[tuple[str, str]]:
    """
    Resume Logic: Read already processed (drug_name, smiles) pairs from the output file.
    """
    if not os.path.exists(out_csv):
        return set()
    try:
        done = pd.read_csv(out_csv, usecols=["drug_name", "smiles"])
        done["drug_name"] = done["drug_name"].astype(str)
        done["smiles"] = done["smiles"].astype(str)
        return set(zip(done["drug_name"], done["smiles"]))
    except Exception as e:
        print(f"Warning: Failed to read existing {out_csv} for resume: {e}")
        return set()

# ==========================================================
# 2) Data Loading and Preprocessing
# ==========================================================
if not os.path.exists(LOOKUP_CSV):
    raise FileNotFoundError(f"Input file not found: {LOOKUP_CSV}")

lookup = pd.read_csv(LOOKUP_CSV)

# Basic cleaning: Remove nulls and whitespace (preserving raw SMILES)
lookup = lookup.dropna(subset=["drug_name", "smiles"]).copy()
lookup["drug_name"] = lookup["drug_name"].astype(str).str.strip()
lookup["smiles"] = lookup["smiles"].astype(str).str.strip()
lookup = lookup[(lookup["drug_name"] != "") & (lookup["smiles"] != "")]

# Deduplicate based on name and SMILES combination
unique_drugs = lookup.drop_duplicates(subset=["drug_name", "smiles"]).reset_index(drop=True)

print(f"Total unique drugs to process: {len(unique_drugs)}")

done_keys = load_done_keys(OUT_CSV)
print(f"Already processed (resuming): {len(done_keys)}")

# ==========================================================
# 3) Main Loop: Generate Descriptions + Embeddings
# ==========================================================
out_exists = os.path.exists(OUT_CSV)

for i, row in unique_drugs.iterrows():
    drug_name = row["drug_name"]
    smiles = row["smiles"]
    key = (drug_name, smiles)

    if key in done_keys:
        continue

    print(f"[{i+1}/{len(unique_drugs)}] Processing: {drug_name}")

    try:
        # Step A: Generate Text Description
        desc = call_with_retries(generate_description, drug_name)
        
        # Step B: Generate Embedding for the Description
        vec = call_with_retries(embed_text, desc)

        out_row = pd.DataFrame([{
            "drug_name": drug_name,
            "smiles": smiles,
            "Description": desc,
            "Embedding": json.dumps(vec),   # Store vector as JSON string
            "EmbeddingDim": len(vec),
            "TextModel": TEXT_MODEL,
            "EmbeddingModel": EMB_MODEL
        }])

        # Save result immediately: Write to CSV to ensure progress is saved if interrupted
        out_row.to_csv(
            OUT_CSV,
            mode="a",
            header=not out_exists,
            index=False
        )
        out_exists = True
        done_keys.add(key)

    except Exception as e:
        print(f"    Critical error processing {drug_name}: {e}")
        continue

    time.sleep(REQUEST_SLEEP_SEC)

print(f"Processing complete. Results saved to: {OUT_CSV}")

In [ ]:
import json
import numpy as np
import pandas as pd

IN_CSV = "drug_desc_embedding.csv"
OUT_NPY = "llm_smiles_embeddings.npy"

df = pd.read_csv(IN_CSV)

# SMILES -> embedding(np.float32)
llm_dict = {}
for _, r in df.iterrows():
    smi = str(r["smiles"]).strip()
    vec = np.array(json.loads(r["Embedding"]), dtype=np.float32)
    llm_dict[smi] = vec

np.save(OUT_NPY, llm_dict, allow_pickle=True)
print(f"Saved {len(llm_dict)} embeddings to {OUT_NPY}")